For faster downloads and for better GPU, use colab extension on VS Code to connect to a remote Colab Instance

# Checking the Non domain-adapted model

ModernBERT-base is used for its benchmarks and size.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, default_data_collator, DataCollatorForLanguageModeling
from datasets import Dataset
import collections
import numpy as np
import math

SEED = 67
CHUNK_SIZE = 128
WWM_PROB = 0.2
BATCH_SIZE = 16 # GPU bound

In [2]:

model_use = "huawei-noah/TinyBERT_General_4L_312D"

tokenizer = AutoTokenizer.from_pretrained(model_use)
model = AutoModelForMaskedLM.from_pretrained(model_use)

config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                               | Status     |  | 
----------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4}.weight | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4}.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight          | UNEXPECTED |  | 
cls.seq_relationship.bias         | UNEXPECTED |  | 
cls.seq_relationship.weight       | UNEXPECTED |  | 
bert.pooler.dense.bias            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Attempt on an arbitrary text on a news body

In [3]:
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"

In [4]:
inputs = tokenizer(test, return_tensors='pt')
outputs = model(**inputs)
logits = outputs.logits

In [5]:
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer.decode([token])}, {test.replace(tokenizer.mask_token, tokenizer.decode(token).strip())}")

##iidae,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week ##iidae as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
##nidae,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week ##nidae as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
##tidae,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week ##tidae as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
##idae,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week ##idae as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodri

## Preprocessing the Dataset

In [6]:
import jsonl
import pandas as pd



titles = []
articles = []

sources = ["abscbn", "gma", "rappler"]
for source in sources:
    with open(f'../scraping/{source}_articles.jsonl', encoding='utf-8', errors="replace") as source_f:
        source_json = jsonl.load(source_f)
        for article in source_json:
            t = article['title']
            b = article['body']
            # print(b)
            titles.append(t)
            articles.append(b)

articles_df = pd.DataFrame({
    "title": titles,
    "body": articles
})

articles_df

,title,body
0,Iran minister heads to Russia as talks remain ...,Iran's foreign minister headed to Russia on Su...
1,Dizon orders immediate declogging of SLEX drai...,Dizon said the DPWH will work together with SL...
2,Marcos to tackle China's gray zone tactics wit...,Marcos said he wants to find out more about 't...
3,11 of 19 Toboso fatalities tested positive for...,"Police Colonel Reynaldo Calaoa, chief of the P..."
4,"Leviste unhappy in Congress, open to other pos...",Speaking at the Kapihan ng Samahang Plaridel i...
...,...,...
3018,[WATCH] Rappler Live Jam: Debonair District,Join us for jazz night on Rappler Live Jam wit...
3019,"Syria army plane crashes in rebel-held town, 3...","BEIRUT, Lebanon – A Syrian military aircraft c..."
3020,Gavina hints at possible roster changes as Rai...,Rain or Shine falls short of its last-ditch ef...
3021,Maria Sharapova learns to box as tennis return...,The 5-time Grand Slam champion also spent her ...


In [7]:
def tokenize_func(data):
    tokenized = tokenizer(data["body"])
    if tokenizer.is_fast:
        tokenized["word_ids"] = [tokenized.word_ids(i) for i in range(len(tokenized["input_ids"]))]
    return tokenized


rappler_ds = Dataset.from_pandas(articles_df)
rappler_ds = rappler_ds.train_test_split(seed=SEED)

tokenized_rappler = rappler_ds.map(
    tokenize_func, batched=True, remove_columns=['title', 'body']
)
tokenized_rappler


Map:   0%|          | 0/2267 [00:00<?, ? examples/s]

Map:   0%|          | 0/756 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 2267
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids'],
        num_rows: 756
    })
})

Concatenate all articles and split each article into texts of length chunk size.

In [8]:
def concatenate_chunk(data):
    concatenated = {key: sum(data[key], []) for key in data}
    total_length = len(concatenated[list(data)[0]])
    # remove last when smaller than chunk size
    total_length = (total_length // CHUNK_SIZE) * CHUNK_SIZE
    # split
    splitted = {
        key: [row[i:i + CHUNK_SIZE] for i in range(0, total_length, CHUNK_SIZE)]
        for key, row in concatenated.items()
    }

    splitted["labels"] = splitted["input_ids"].copy()

    return splitted

In [9]:
rappler_chunked = tokenized_rappler.map(concatenate_chunk, batched=True)
rappler_chunked

Map:   0%|          | 0/2267 [00:00<?, ? examples/s]

Map:   0%|          | 0/756 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 11563
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 3844
    })
})

In [10]:
tokenizer.decode(rappler_chunked["train"][0]["input_ids"])

"[CLS] overcoming a deficit down the stretch seems to be lyceum of the philippines university ’ s forte this season as they came back from 12 points down to defeat colegio de san juan de letran, 85 - 79. the pirates, who are now on a four - game win streak, outscored the knights 29 - 11 in the final canto to add to the defending champions ’ woes this year. when asked about how his squad is able to come back despite being against the ropes time and time again, lyceum head coach gilbert malabanan attributes this ' never say die ' mentality to their strength in numbers"

## Fine-tuning

In [11]:
# perform whole word masking, not based on tokens, rather on "whole words" based on the collated words gathered during preprocessing
# Adapted from Hugging Face Documentation https://huggingface.co/learn/llm-course/chapter7/3
def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # create a map for a word to list of indices in tokenizer
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)
        
        # mask

        mask = np.random.binomial(1, WWM_PROB, (len(mapping), ))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels
    
    return default_data_collator(features)

### Training

In [12]:
logging_steps = len(rappler_chunked["train"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir=f"{model_use.split('/')[-1]}-finetuned-all",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=BATCH_SIZE,
    per_device_train_batch_size=BATCH_SIZE,
    fp16=True,
    logging_steps=logging_steps
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=rappler_chunked["train"],
    eval_dataset=rappler_chunked["test"],
    data_collator=data_collator,
)

In [14]:
eval_results = trainer.evaluate()
print(f"Perplexity before fine-tuning: {math.exp(eval_results['eval_loss'])}")

Training Loss,Validation Loss,Epoch
No log,10.585794,0


Perplexity before fine-tuning: 39568.73039606741


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,6.068799
2,No log,5.746805
3,No log,5.651639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2169, training_loss=6.182543348749424, metrics={'train_runtime': 142.2114, 'train_samples_per_second': 243.926, 'train_steps_per_second': 15.252, 'total_flos': 125164447962624.0, 'train_loss': 6.182543348749424, 'epoch': 3.0})

In [16]:
eval_results = trainer.evaluate()
print(f"Perplexity after fine-tuning: {math.exp(eval_results['eval_loss'])}")

Training Loss,Validation Loss,Epoch
No log,5.675218,3


Perplexity after fine-tuning: 291.5519206350998


In [17]:
trainer.save_model() 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch


tokenizer_test = AutoTokenizer.from_pretrained('./ModernBERT-base-finetuned-rappler')
model_test = AutoModelForMaskedLM.from_pretrained('./ModernBERT-base-finetuned-rappler')


test = "CS Week 2025 [MASK] with a blast as students conquer the halls with glee"
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"
test_input = tokenizer_test(test, return_tensors="pt")

test_output = model_test(**test_input)
test_logits = test_output.logits


mask_token_index = torch.where(test_input["input_ids"] == tokenizer_test.mask_token_id)[1]
mask_token_logits = test_logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer_test.decode([token])}, {test.replace(tokenizer_test.mask_token, tokenizer_test.decode(token).strip())}")

top_10 = torch.argsort(mask_token_logits, dim = 1, descending=True)
print(top_10)

for t in top_10.tolist():
    print(t)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

 stint,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stint as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 tenure,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week tenure as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 term,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week term as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 stay,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stay as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s